In [18]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from numba import njit, prange
from pathlib import Path
import warnings

# Désactive les alertes matplotlib mineures
warnings.filterwarnings('ignore') 

# Configuration visuelle
%matplotlib inline
sns.set_theme(style="white")
plt.rcParams['figure.dpi'] = 100

# --- 1. CHARGEMENT DE LA MATRICE D'ORACLE ---
GAP_OFFSET = 600
project_root = Path.cwd().parent 
import sys
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from mpp_project.oracle_dp import N_PLAYERS  # effectifs : source unique = oracle_dp
data_path_npy = project_root / "data" / "expected_V_phases_finales.npy"

try:
    V_all_matches = np.load(data_path_npy)
    print(f"✅ Matrice d'Oracle chargée : {V_all_matches.shape}")
except FileNotFoundError:
    print(f"❌ Erreur : Fichier introuvable à {data_path_npy}")


# --- 2. LE MOTEUR DE CALCUL (NUMBA) ---
@njit
def terminal_val(g1, g2):
    """Win Rate absolu à la fin de la compétition."""
    if g1 > 0 and g2 > 0: return 1.0
    elif (g1 == 0 and g2 > 0) or (g1 > 0 and g2 == 0): return 0.5
    elif g1 == 0 and g2 == 0: return 1.0 / 3.0
    else: return 0.0

@njit(parallel=True)
def compute_2d_tactical_map(V_all, match_idx, probas, gains, crowds, has_booster, gap_min, gap_max):
    """Calcule la stratégie optimale pour tous les couples (Gap 1, Gap 2) sur un match précis."""
    len_gaps = gap_max - gap_min + 1
    strategy = np.zeros((len_gaps, len_gaps), dtype=np.float32)
    is_boost_optimal = np.zeros((len_gaps, len_gaps), dtype=np.bool_)
    
    # Physique de l'élastique du peloton MPP
    N_MATCHES, N_MATCHES_POULES = 104, 72  # N_PLAYERS importé d'oracle_dp
    global_t = N_MATCHES_POULES + match_idx
    time_fraction = 1.0 - (global_t / N_MATCHES)
    N_eff = 1.0 + (N_PLAYERS - 3.0) * (time_fraction ** 5.50)
    
    for i in prange(len_gaps):
        g1 = gap_min + i
        for j in range(len_gaps):
            g2 = gap_min + j
            
            # État mathématiquement impossible (Leader a plus de retard que le Second)
            if g1 > g2:
                strategy[i, j] = np.nan
                is_boost_optimal[i, j] = False
                continue
                
            best_expected_v = -1.0
            best_action = 0
            boost_triggered = False
            
            for agent_action in range(3):
                expected_v_keep = 0.0
                expected_v_use = 0.0
                
                for out in range(3):
                    p_out = probas[out]
                    a_g = gains[out] if agent_action == out else 0
                    a_g_boost = (gains[out] * 2) if agent_action == out else 0
                    prob_pack_hits = 1.0 - (1.0 - crowds[out]) ** N_eff
                    
                    for bob_act in range(3):
                        p_bob = crowds[bob_act]
                        bob_g = gains[out] if bob_act == out else 0
                        pack_g_hit = gains[out]
                        pack_g_miss = 0
                        
                        # --- SCÉNARIO A : Le Peloton trouve ---
                        jp_hit = p_out * p_bob * prob_pack_hits
                        
                        # Univers "Garder le booster"
                        ng1_h_k = max(-600, min(400, min(g1 + a_g - bob_g, g2 + a_g - pack_g_hit)))
                        ng2_h_k = max(-600, min(400, max(g1 + a_g - bob_g, g2 + a_g - pack_g_hit)))
                        # Univers "Utiliser le booster"
                        ng1_h_u = max(-600, min(400, min(g1 + a_g_boost - bob_g, g2 + a_g_boost - pack_g_hit)))
                        ng2_h_u = max(-600, min(400, max(g1 + a_g_boost - bob_g, g2 + a_g_boost - pack_g_hit)))
                        
                        if match_idx < V_all.shape[0] - 1:
                            v_k_hit = V_all[match_idx+1, ng1_h_k + 600, ng2_h_k + 600, 1]
                            v_u_hit = V_all[match_idx+1, ng1_h_u + 600, ng2_h_u + 600, 0]
                        else:
                            v_k_hit = terminal_val(ng1_h_k, ng2_h_k)
                            v_u_hit = terminal_val(ng1_h_u, ng2_h_u)
                            
                        expected_v_keep += jp_hit * v_k_hit
                        expected_v_use += jp_hit * v_u_hit
                        
                        # --- SCÉNARIO B : Le Peloton rate ---
                        jp_miss = p_out * p_bob * (1.0 - prob_pack_hits)
                        
                        ng1_m_k = max(-600, min(400, min(g1 + a_g - bob_g, g2 + a_g - pack_g_miss)))
                        ng2_m_k = max(-600, min(400, max(g1 + a_g - bob_g, g2 + a_g - pack_g_miss)))
                        
                        ng1_m_u = max(-600, min(400, min(g1 + a_g_boost - bob_g, g2 + a_g_boost - pack_g_miss)))
                        ng2_m_u = max(-600, min(400, max(g1 + a_g_boost - bob_g, g2 + a_g_boost - pack_g_miss)))
                        
                        if match_idx < V_all.shape[0] - 1:
                            v_k_miss = V_all[match_idx+1, ng1_m_k + 600, ng2_m_k + 600, 1]
                            v_u_miss = V_all[match_idx+1, ng1_m_u + 600, ng2_m_u + 600, 0]
                        else:
                            v_k_miss = terminal_val(ng1_m_k, ng2_m_k)
                            v_u_miss = terminal_val(ng1_m_u, ng2_m_u)
                            
                        expected_v_keep += jp_miss * v_k_miss
                        expected_v_use += jp_miss * v_u_miss
                
                # Choix de l'action maximisant l'Espérance de WR global
                if expected_v_keep > best_expected_v:
                    best_expected_v = expected_v_keep
                    best_action = agent_action
                    boost_triggered = False
                    
                if has_booster == 1 and expected_v_use > best_expected_v:
                    best_expected_v = expected_v_use
                    best_action = agent_action
                    boost_triggered = True
                    
            strategy[i, j] = best_action
            is_boost_optimal[i, j] = boost_triggered
            
    return strategy, is_boost_optimal


# --- 3. L'INTERFACE GRAPHIQUE (DASHBOARD TACTIQUE) CORRIGÉE ---
def plot_tactical_map(match_idx, cote_1, cote_N, cote_2, c1, cN, g1, gN, g2, has_booster):
    # 1. Conversion des Cotes en Probabilités Pures (Suppression de la marge)
    raw_p1 = 1.0 / cote_1
    raw_pN = 1.0 / cote_N
    raw_p2 = 1.0 / cote_2
    
    margin = raw_p1 + raw_pN + raw_p2
    p1 = raw_p1 / margin
    pN = raw_pN / margin
    p2 = raw_p2 / margin
    
    # 2. Vérification et calcul du Crowd restant
    c2 = round(1.0 - c1 - cN, 2)
    if c2 < 0:
        print("⚠️ Répartition de la foule (Crowd) invalide ! La somme de 1 et N dépasse 100%.")
        return
        
    probas = np.array([p1, pN, p2], dtype=np.float32)
    gains = np.array([g1, gN, g2], dtype=np.int32)
    crowds = np.array([c1, cN, c2], dtype=np.float32)
    
    # Bornes de visualisation
    gap_min, gap_max = -200, 100
    
    # Appel de l'Oracle
    strategy, is_boost_optimal = compute_2d_tactical_map(
        V_all_matches, match_idx, probas, gains, crowds, has_booster, gap_min, gap_max
    )
    
    # Esthétique
    cmap = ListedColormap(['#2ecc71', '#f1c40f', '#e74c3c'])
    bounds = [-0.5, 0.5, 1.5, 2.5]
    norm = BoundaryNorm(bounds, cmap.N)
    
    fig, ax = plt.subplots(figsize=(11, 8))
    X = np.arange(gap_min, gap_max + 2)
    Y = np.arange(gap_min, gap_max + 2)
    
    # Fond coloré des équipes
    c = ax.pcolormesh(X, Y, strategy, cmap=cmap, norm=norm, alpha=0.85)
    
    # Hachures pour l'utilisation du Booster
    if has_booster == 1:
        boost_mask = np.where(is_boost_optimal, 1, 0)
        boost_mask_ma = np.ma.masked_where(boost_mask == 0, boost_mask)
        
        # CORRECTION ICI : Remplacement de fill=False par facecolor='none'
        ax.pcolor(X, Y, boost_mask_ma, hatch='\\\\\\\\', facecolor='none', edgecolor='#1f77b4', linewidth=0)
    
    # Ligne d'égalité
    ax.plot([gap_min, gap_max], [gap_min, gap_max], color='black', linestyle='--', linewidth=1.5)
    
    match_reel = match_idx + 72
    ax.set_title(f"Plan de Frappe Tactique - Match {match_reel}\n"
                 f"[Probas : 1={p1*100:.1f}% | N={pN*100:.1f}% | 2={p2*100:.1f}%] (Marge Bookmaker : {(margin-1)*100:.1f}%)", 
                 fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel("Écart de points avec le Second (Gap 2)", fontsize=11)
    ax.set_ylabel("Écart de points avec le Leader (Gap 1)", fontsize=11)
    
    # Légende
    import matplotlib.patches as mpatches
    legend_elements = [
        mpatches.Patch(color='#2ecc71', label='Jouer Favori (1)'),
        mpatches.Patch(color='#f1c40f', label='Jouer Nul (N)'),
        mpatches.Patch(color='#e74c3c', label='Jouer Outsider (2)'),
        plt.Line2D([0], [0], color='black', linestyle='--', label="Frontière (Gap 1 = Gap 2)")
    ]
    if has_booster == 1:
        legend_elements.append(mpatches.Patch(facecolor='none', edgecolor='#1f77b4', hatch='\\\\\\\\', label='Zone optimale pour claquer le x2'))
        
    ax.legend(handles=legend_elements, loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
    plt.tight_layout()
    plt.show()

print("\n--- INITIALISATION DU BAC À SABLE TACTIQUE 2D ---")
interact(
    plot_tactical_map,
    match_idx=IntSlider(min=0, max=31, step=1, value=15, description="Match (0=J72)"),
    has_booster=Dropdown(options=[('Booster Dispo (1)', 1), ('Booster Utilisé (0)', 0)], value=1, description='Mon Booster'),
    cote_1=FloatSlider(min=1.1, max=5.0, step=0.05, value=1.85, description="Cote Favori"),
    cote_N=FloatSlider(min=2.0, max=10.0, step=0.05, value=3.40, description="Cote Nul"),
    cote_2=FloatSlider(min=1.5, max=15.0, step=0.05, value=4.20, description="Cote Outsider"),
    c1=FloatSlider(min=0.1, max=0.9, step=0.05, value=0.65, description="Crowd % (1)"),
    cN=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.15, description="Crowd % (N)"),
    g1=IntSlider(min=10, max=100, step=5, value=30, description="Points MPP (1)"),
    gN=IntSlider(min=30, max=150, step=5, value=60, description="Points MPP (N)"),
    g2=IntSlider(min=30, max=200, step=5, value=90, description="Points MPP (2)")
);

✅ Matrice d'Oracle chargée : (32, 1001, 1001, 2)

--- INITIALISATION DU BAC À SABLE TACTIQUE 2D ---


interactive(children=(IntSlider(value=15, description='Match (0=J72)', max=31), FloatSlider(value=1.85, descri…